In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# إنشاء الجلسة وربطها بالمسارات الصحيحة داخل حاوية itversity
spark = SparkSession.builder \
    .appName("Lab2-Temperature-Streaming") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/data/spark-assignments/lab2_temperature/hive_warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

# تقليل الـ partitions لتسريع المعالجة التفاعلية محلياً
spark.conf.set("spark.sql.shuffle.partitions", "2")

print("⚡ تم تشغيل SparkSession بنجاح داخل الحاوية!")
spark

⚡ تم تشغيل SparkSession بنجاح داخل الحاوية!


In [6]:
# قراءة ملف واحد بشكل عادي لاستكشاف الحقول ونوع البيانات الداخلي
sample_df = spark.read.json("file:///data/spark-assignments/lab2_temperature/json_input_dir/batch1.json")

print("📋 الهيكل الفعلي للملف المرفق (Actual JSON Schema):")
sample_df.printSchema()

print("\n👀 عينة من البيانات الداخلية للملف:")
sample_df.show(5, truncate=False)

📋 الهيكل الفعلي للملف المرفق (Actual JSON Schema):
root
 |-- _corrupt_record: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- temperature: double (nullable = true)


👀 عينة من البيانات الداخلية للملف:
+---------------+-------+-------------------+-----------+
|_corrupt_record|country|event_timestamp    |temperature|
+---------------+-------+-------------------+-----------+
|[              |null   |null               |null       |
|null           |USA    |2024-01-15 10:05:23|12.5       |
|null           |UK     |2024-01-15 10:08:45|5.2        |
|null           |Japan  |2024-01-15 10:12:30|8.7        |
|null           |Germany|2024-01-15 10:15:00|3.4        |
+---------------+-------+-------------------+-----------+
only showing top 5 rows



In [7]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# 1. استخدام المسار المحلي للحاوية
INPUT_DIR = "file:///data/spark-assignments/lab2_temperature/json_input_dir"

# 2. المخطط المحدث بناءً على أسماء الحقول الحقيقية للبيانات
json_schema = StructType([
    StructField("event_timestamp", TimestampType(), True), # تحويل تلقائي من string إلى timestamp
    StructField("country", StringType(), True),
    StructField("temperature", DoubleType(), True)
])

# 3. قراءة التدفق مع تفعيل .option("multiLine", "true") لقراءة المصفوفات بسلام
input_stream_df = spark.readStream \
    .schema(json_schema) \
    .option("multiLine", "true") \
    .json(INPUT_DIR)

print("📊 Inbound Stream Schema Updated Successfully:")
input_stream_df.printSchema()

📊 Inbound Stream Schema Updated Successfully:
root
 |-- event_timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)



In [8]:
from pyspark.sql.functions import col, window, avg

# تطبيق الـ Watermarking والتجميع عبر نافذة زمنية مدتها 15 دقيقة 
# إهمال البيانات المتأخرة لأكثر من 10 دقائق بناءً على العمود event_timestamp [cite: 4]
aggregated_df = input_stream_df \
    .withWatermark("event_timestamp", "10 minutes") \
    .groupBy(
        window(col("event_timestamp"), "15 minutes"),
        col("country")
    ) \
    .agg(avg("temperature").alias("average_temperature")) # [cite: 3]

print("📐 Aggregated Data Schema Ready:")
aggregated_df.printSchema()

📐 Aggregated Data Schema Ready:
root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- average_temperature: double (nullable = true)



In [10]:
# 1. تحديد مسارات المخرجات ونقاط الفحص (Checkpoints) على النظام المحلي
FILE_OUTPUT_DIR = "file:///data/spark-assignments/lab2_temperature/output_files/temperature_report"
CHECKPOINT_FILE = "file:///data/spark-assignments/lab2_temperature/checkpoints/file_target"

# الوجهة الثانية البديلة (نظام الملفات - مسار منفصل) تحقيقاً للمطلب 
SECONDARY_OUTPUT_DIR = "file:///data/spark-assignments/lab2_temperature/output_files/secondary_temperature_report"
CHECKPOINT_SECONDARY = "file:///data/spark-assignments/lab2_temperature/checkpoints/secondary_target"

print("🚀 Starting Stream Processing... Watching for new JSON files in json_input_dir...")

# الوجهة الأولى: نظام الملفات الرئيسي (Parquet) 
file_query = aggregated_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", FILE_OUTPUT_DIR) \
    .option("checkpointLocation", CHECKPOINT_FILE) \
    .start()

# الوجهة الثانية: نظام الملفات الفرعي البديل لـ Hive (Parquet) 
secondary_query = aggregated_df.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", SECONDARY_OUTPUT_DIR) \
    .option("checkpointLocation", CHECKPOINT_SECONDARY) \
    .start()

🚀 Starting Stream Processing... Watching for new JSON files in json_input_dir...
